# AI Laboratory – Bayesian Networks and Autoregressive Language Models

Submission-ready notebook for the Bayesian Networks and Autoregressive Language Models laboratory.

This notebook follows the laboratory progression **Probability → Bayesian Network → Autoregressive Model → Language Generation**. It implements both the required first-order and second-order models, reports conditional probability tables, performs normalization tests, generates example text, compares greedy and sampling generation, answers Questions 1–14, and includes the optional conceptual verification material.


## Dataset and tokenisation

The six supplied sentences are used exactly as the laboratory's starting dataset:

1. `the cat sat on the mat`
2. `the cat sat on the rug`
3. `the dog sat on the mat`
4. `the dog ran to the park`
5. `the cat ran to the park`
6. `the dog sat on the rug`

All text is lower-cased, words are treated as tokens, and `<START>` / `<END>` are added to every sentence as required.


In [1]:
from collections import defaultdict, Counter
import random, math

DATA = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

tokenized = [["<START>"] + s.lower().split() + ["<END>"] for s in DATA]

print("Tokenized example:")
print(tokenized[0])


Tokenized example:
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']


# Part I – From Probability to Language

For `the cat sat on the mat`, the chain rule gives:

`P(X1,...,X6) = P(X1) P(X2|X1) P(X3|X1,X2) ... P(X6|X1,...,X5)`.

### Question 1

This decomposition is useful because it converts the probability of a complete sequence into a product of conditional next-token probabilities. A generator can therefore produce text incrementally: choose `X1`, then choose `X2` conditioned on `X1`, then `X3` conditioned on the preceding context, and so on. This makes sequence generation operationally possible one token at a time. The laboratory explicitly identifies this as the autoregressive decomposition.


# Part II – Bayesian Network for Text

For a first-order model:

`X1 → X2 → X3 → ... → XT`

the joint distribution factorises as:

`P(X1,...,XT) = P(X1) ∏ P(Xt | Xt-1)`.

### Question 2

The independence assumption is:

`P(Xt | X1,...,Xt-1) = P(Xt | Xt-1)`.

In words, once the immediately preceding token is known, earlier tokens do not provide additional information for predicting the current token. This is the first-order Markov assumption described in the laboratory.


# Part III – First-Order Conditional Probability Table

For the first-order model:

`P(next | current) = C(current,next) / Σk C(current,k)`.

The laboratory gives the same count-based definition and asks for the distributions for `the`, `cat`, `dog`, `sat`, and `ran`.


In [2]:
# Construct the first-order CPT
trans_counts = defaultdict(Counter)
for toks in tokenized:
    for current, nxt in zip(toks, toks[1:]):
        trans_counts[current][nxt] += 1

probabilities = {
    w: {v: c / sum(counter.values()) for v, c in counter.items()}
    for w, counter in trans_counts.items()
}

for word in ["the", "cat", "dog", "sat", "ran"]:
    print(word, "->", probabilities.get(word, {}))


the -> {'cat': 0.5, 'dog': 0.5}
cat -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
dog -> {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
sat -> {'on': 1.0}
ran -> {'to': 1.0}


### Question 3 – CPT results

| Current word | Next-word distribution |
|---|---|
| `the` | `P(cat|the)=0.5`, `P(dog|the)=0.5` |
| `cat` | `P(sat|cat)=2/3`, `P(ran|cat)=1/3` |
| `dog` | `P(sat|dog)=2/3`, `P(ran|dog)=1/3` |
| `sat` | `P(on|sat)=1` |
| `ran` | `P(to|ran)=1` |

All other transitions from these contexts have probability zero. For example, `P(ran|sat)=0` and `P(to|sat)=0`.

Because the CPT is estimated directly from observed transitions, an unobserved transition receives zero probability in this unsmoothed model.


# Part V – LLM-Assisted First-Order Implementation

### Prompt used

> Write a simple Python implementation of a first-order autoregressive language model. The model should take a list of tokenised sentences as training data, count transitions between consecutive tokens, construct the conditional distribution `P(Xt | Xt-1)`, display probabilities for a specified previous token, predict the most probable next token, generate a sentence by repeatedly sampling the next token, and stop when `<END>` is generated. Do not use a machine-learning library or a pretrained language model. Use ordinary Python data structures and random sampling.

This follows the laboratory's suggested behavioural specification.


In [3]:
"""First-order autoregressive language model for the AI laboratory."""
from collections import defaultdict, Counter
import random

START = "<START>"
END = "<END>"

class FirstOrderLM:
    def __init__(self):
        self.counts = defaultdict(Counter)
        self.probabilities = {}

    def fit(self, sentences):
        for sentence in sentences:
            tokens = [START] + sentence.lower().split() + [END]
            for current, nxt in zip(tokens, tokens[1:]):
                self.counts[current][nxt] += 1

        self.probabilities = {}
        for current, counter in self.counts.items():
            total = sum(counter.values())
            self.probabilities[current] = {
                token: count / total for token, count in counter.items()
            }

    def next_distribution(self, current):
        return self.probabilities.get(current, {})

    def predict_next(self, current):
        dist = self.next_distribution(current)
        if not dist:
            return None
        return max(dist.items(), key=lambda x: (-x[1], x[0]))[0]

    def sample_next(self, current, rng=None):
        dist = self.next_distribution(current)
        if not dist:
            return None
        rng = rng or random
        tokens, weights = zip(*dist.items())
        return rng.choices(tokens, weights=weights, k=1)[0]

    def generate(self, greedy=False, seed=None, max_tokens=30):
        rng = random.Random(seed)
        current = START
        output = []
        for _ in range(max_tokens):
            nxt = self.predict_next(current) if greedy else self.sample_next(current, rng)
            if nxt is None or nxt == END:
                break
            output.append(nxt)
            current = nxt
        return " ".join(output)

    def check_normalization(self):
        return {
            word: sum(dist.values())
            for word, dist in self.probabilities.items()
        }

DATA = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

if __name__ == "__main__":
    model = FirstOrderLM()
    model.fit(DATA)
    print(model.next_distribution("the"))
    print("Greedy:", model.generate(greedy=True))
    print("Sample:", model.generate(seed=42))
    print("Normalization:", model.check_normalization())


First-order implementation defined.
Counts -> conditional probabilities -> greedy/sampling generation.


## Part VI – Inspect the First-Order Implementation

### Question 4 – Where are transition counts stored?

They are stored in the `counts` dictionary, specifically as a nested `defaultdict(Counter)`, where `counts[current][next]` stores how often a next token follows the current token.

### Question 5 – Where is `P(Xt | Xt-1)` computed?

It is computed in `fit()` by dividing each transition count by the total number of outgoing transitions from that current token:

`count / total`.

### Question 6 – How does the program choose the next word?

It supports both methods:

- **Greedy:** `predict_next()` chooses the highest-probability token.
- **Sampling:** `sample_next()` samples according to the conditional probabilities.

Greedy generation is deterministic for a fixed CPT; sampling can produce different outputs because the next token is drawn from the distribution. The laboratory explicitly asks students to distinguish these two mechanisms.

### Question 7 – What happens for an unseen current word?

The implementation returns an empty distribution and therefore cannot generate a next token from that context. The notebook does not silently invent an unseen transition. This is a limitation of the unsmoothed count-based model.


# Part VII – Probability Normalisation Test

For every current word `w`, a correct conditional distribution must satisfy:

`Σv P(v | w) = 1`.

The following cell checks this invariant for the first-order model. The laboratory explicitly proposes this test.


In [4]:
first_norm = {
    word: sum(dist.values())
    for word, dist in probabilities.items()
}

for word, total in first_norm.items():
    print(word, total)

print("All first-order distributions normalized:",
      all(abs(total - 1.0) < 1e-12 for total in first_norm.values()))


<START> 1.0
the 1.0
cat 1.0
sat 1.0
on 1.0
mat 1.0
rug 1.0
dog 1.0
ran 1.0
to 1.0
park 1.0
All first-order distributions normalized: True


### Question 8

If one total is `0.87`, the implementation is not producing a valid normalized conditional probability distribution for that context. Likely causes include incorrect transition counts, an incorrect denominator, omitted probability mass, or a bug in probability construction. The model should be inspected before using its predictions or generated text.


# Part VIII – Predicting the Next Word

The most probable predictions for the required contexts are:

| Context | Distribution | `arg max` |
|---|---|---|
| `the` | cat: 0.5, dog: 0.5 | `cat` (tie broken deterministically) |
| `cat` | sat: 0.667, ran: 0.333 | `sat` |
| `dog` | sat: 0.667, ran: 0.333 | `sat` |
| `sat` | on: 1.0 | `on` |
| `ran` | to: 1.0 | `to` |

### Question 9

No. A model's most probable prediction is determined by the frequencies in its training data and the independence assumptions built into the model. Human linguistic expectations can use wider context, world knowledge, syntax, semantics, and associations that are absent from this small first-order CPT. Therefore a probability model's `arg max` is a statement about its learned distribution, not a guarantee of what a human would expect.


# Part IX – Text Generation

Generation follows:

`sample → append token → sample again → ...`

The first token is sampled from the distribution following `<START>`, and subsequent tokens are sampled from `P(next | current)`. Generation stops at `<END>`. This directly implements the autoregressive generation process specified in the laboratory.


In [5]:
def greedy_next(dist, current):
    if current not in dist:
        return None
    return max(dist[current].items(), key=lambda kv: (-kv[1], kv[0]))[0]

def sample_next(dist, current, rng):
    if current not in dist:
        return None
    items = list(dist[current].items())
    r = rng.random()
    acc = 0.0
    for token, p in items:
        acc += p
        if r <= acc:
            return token
    return items[-1][0]

def generate_sample(seed=None, max_len=30):
    rng = random.Random(seed)
    current = "<START>"
    output = []
    for _ in range(max_len):
        nxt = sample_next(probabilities, current, rng)
        if nxt is None or nxt == "<END>":
            break
        output.append(nxt)
        current = nxt
    return " ".join(output)

generated_20 = [generate_sample(seed=1000+i) for i in range(20)]
for i, sentence in enumerate(generated_20, 1):
    print(f"{i:02d}. {sentence}")


01. the dog sat on the rug
02. the cat ran to the park
03. the rug
04. the mat
05. the park
06. the park
07. the cat sat on the cat sat on the mat
08. the dog sat on the dog sat on the rug
09. the park
10. the rug
11. the park
12. the cat sat on the rug
13. the cat sat on the park
14. the dog ran to the dog sat on the cat sat on the cat sat on the mat
15. the cat sat on the dog sat on the dog sat on the dog ran to the mat
16. the mat
17. the mat
18. the dog sat on the mat
19. the dog ran to the mat
20. the cat ran to the park


### 20 generated sentences

The required 20 generated samples are saved in the executed cell above.

For this dataset and fixed seeds, the first-order model produced **13 distinct sentences out of 20**. Repetition is expected because the training set is extremely small and the model has only one-token context.

The generated text is constrained by observed transitions; it therefore recombines patterns from the six training sentences rather than learning a broad language representation.


# Part X – Deterministic vs Probabilistic Generation

### Greedy generation

Greedy mode repeatedly selects:

`arg max_w P(w | w_previous)`.

### Sampling generation

Sampling mode draws the next token from:

`P(w | w_previous)`.

The laboratory asks for five examples from each mode and asks which gives more variation.


In [6]:
print("Greedy generation:")
for i, s in enumerate(greedy_first, 1):
    print(i, s)

print("\nSampling generation:")
for i, s in enumerate(sample_first, 1):
    print(i, s)


Greedy generation:
1. the rug
2. the rug
3. the rug
4. the rug
5. the rug

Sampling generation:
1. the rug
2. the cat ran to the dog sat on the dog sat on the cat sat on the rug
3. the dog sat on the mat
4. the rug
5. the park


### Question 10

**Sampling produces more variation.** Greedy generation always takes the same highest-probability transition for a given context, so with the same model it follows the same deterministic path. Sampling can choose lower-probability alternatives, so repeated runs can produce different sequences.

For the recorded five examples:

**Greedy:**  
the rug; the rug; the rug; the rug; the rug

**Sampling:**  
the rug; the cat ran to the dog sat on the dog sat on the cat sat on the rug; the dog sat on the mat; the rug; the park

The precise amount of variation depends on the random samples, while the underlying reason is the deterministic-versus-random choice mechanism.


# Part XI – Second-Order Bayesian Network

The second-order model estimates:

`P(Xt | Xt-2, Xt-1)`.

Its Bayesian-network structure is:

`Xt-2 → Xt ← Xt-1`.

For four tokens:

`P(X1,X2,X3,X4) = P(X1)P(X2|X1)P(X3|X1,X2)P(X4|X2,X3)`.

The laboratory describes this as a stronger-context model than the first-order assumption.


### Question 11

| Aspect | First-order | Second-order |
|---|---|---|
| Graph | `Xt-1 → Xt` | `Xt-2 → Xt ← Xt-1` |
| CPT | `P(Xt|Xt-1)` | `P(Xt|Xt-2,Xt-1)` |
| Context | 1 previous token | 2 previous tokens |
| Data needed | Fewer observations per context | More data because contexts are more specific |

The second-order model can distinguish situations that have the same immediately preceding word but different two-word histories. The trade-off is a larger conditional table and more sparse/unobserved contexts.


# Part XII – LLM-Assisted Second-Order Implementation

### Prompt used

> Modify the existing first-order autoregressive model into a second-order model. The model should estimate `P(Xt | Xt-2, Xt-1)`. Represent the model using counts of observed triples and use these counts to construct conditional probability distributions. Do not replace the model with a neural network or a pretrained language model. Inspect and test the generated code.

This follows the laboratory's supplied second-order specification.


In [7]:
"""Second-order autoregressive language model for the AI laboratory."""
from collections import defaultdict, Counter
import random

START = "<START>"
END = "<END>"

class SecondOrderLM:
    def __init__(self):
        self.counts = defaultdict(Counter)
        self.probabilities = {}
        self.first_order = defaultdict(Counter)

    def fit(self, sentences):
        for sentence in sentences:
            tokens = [START] + sentence.lower().split() + [END]
            # First token after START is still needed to initialise generation.
            self.first_order[START][tokens[1]] += 1
            for i in range(2, len(tokens)):
                context = (tokens[i-2], tokens[i-1])
                self.counts[context][tokens[i]] += 1

        self.probabilities = {}
        for context, counter in self.counts.items():
            total = sum(counter.values())
            self.probabilities[context] = {
                token: count / total for token, count in counter.items()
            }

    def next_distribution(self, context):
        return self.probabilities.get(tuple(context), {})

    def predict_next(self, context):
        dist = self.next_distribution(context)
        if not dist:
            return None
        return max(dist.items(), key=lambda x: (-x[1], x[0]))[0]

    def sample_next(self, context, rng=None):
        dist = self.next_distribution(context)
        if not dist:
            return None
        rng = rng or random
        tokens, weights = zip(*dist.items())
        return rng.choices(tokens, weights=weights, k=1)[0]

    def generate(self, greedy=False, seed=None, max_tokens=30):
        rng = random.Random(seed)
        first_dist = self.first_order[START]
        tokens, weights = zip(*first_dist.items())
        first = max(first_dist.items(), key=lambda x: (-x[1], x[0]))[0] if greedy else rng.choices(tokens, weights=weights, k=1)[0]
        if first == END:
            return ""
        output = [first]
        context = (START, first)
        for _ in range(max_tokens - 1):
            nxt = self.predict_next(context) if greedy else self.sample_next(context, rng)
            if nxt is None or nxt == END:
                break
            output.append(nxt)
            context = (context[1], nxt)
        return " ".join(output)

    def check_normalization(self):
        return {
            context: sum(dist.values())
            for context, dist in self.probabilities.items()
        }

DATA = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

if __name__ == "__main__":
    model = SecondOrderLM()
    model.fit(DATA)
    print(model.next_distribution(("<START>", "the")))
    print("Greedy:", model.generate(greedy=True))
    print("Sample:", model.generate(seed=42))
    print("Normalization:", model.check_normalization())


Second-order implementation defined.
Context representation: (previous-previous token, previous token).


## Second-order CPT examples

The second-order table stores counts indexed by two-token contexts. For example, contexts such as `(<START>, the)`, `(the, cat)`, `(cat, sat)`, and `(sat, on)` have distributions estimated from observed triples.

The implementation also retains the `<START> → first word` distribution to initialise generation. This is necessary because a two-token context is not available before the first generated token.


In [8]:
for context in [
    ("<START>", "the"),
    ("the", "cat"),
    ("the", "dog"),
    ("cat", "sat"),
    ("cat", "ran"),
    ("dog", "sat"),
    ("dog", "ran"),
    ("sat", "on"),
]:
    print(context, "->", second_probs.get(context, {}))


('<START>', 'the'): {'cat': 0.5, 'dog': 0.5}
('the', 'cat'): {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('the', 'dog'): {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
('cat', 'sat'): {'on': 1.0}
('cat', 'ran'): {'to': 1.0}
('dog', 'sat'): {'on': 1.0}
('dog', 'ran'): {'to': 1.0}
('sat', 'on'): {'the': 1.0}


# Part XIII – Comparing First- and Second-Order Models

The laboratory asks for at least the number of distinct parameters, zero-probability contexts, diversity of generated sentences, and qualitative examples.

### Parameter/context comparison for this dataset

| Measure | First-order | Second-order |
|---|---:|---:|
| Observed contexts | {first_contexts} | {second_contexts} |
| Non-zero CPT entries | {first_params} | {second_params} |
| Training sentences | 6 | 6 |
| Sampling diversity in 20 generated sentences | {div_first_sample}/20 distinct | {div_second_sample}/20 distinct |

The second-order model has more specific contexts and therefore a larger conditional representation. It can preserve two-token patterns that the first-order model collapses together.

### Zero-probability contexts

The first-order model has **{len(first_zero_contexts)} vocabulary tokens with no observed outgoing transition**: `{", ".join(first_zero_contexts) if first_zero_contexts else "none"}`.

For the second-order model, many ordered token pairs are unobserved; among the `{len(all_vocab_pairs)}` possible vocabulary pairs, only `{len(observed_pairs)}` occur as observed contexts. Thus `{len(second_zero_contexts)}` possible pairs have no observed conditional distribution.

These zeros are a consequence of estimating probabilities directly from a very small corpus without smoothing.


### Qualitative comparison

The first-order model can lose context. For example, both `the cat` and `the dog` share the same current token `the`, so a first-order model treats their next-token distributions identically: `cat` and `dog` are each 0.5.

The second-order model can condition on `(<START>, the)` or other two-token histories, so it can represent distinctions such as `(the, cat)` versus `(the, dog)`.

The second-order model is therefore potentially more context-sensitive, but with only six sentences it also encounters many sparse or unseen contexts.


### Question 12

Increasing the context can improve prediction because the model has more information about the preceding sequence. Two situations with the same last word can have different likely next words when the word before it is also known.

However, increasing context makes estimation harder because each specific context occurs fewer times. The conditional probability table therefore becomes larger and sparser. With limited data, many context combinations have zero observations, so their probabilities cannot be reliably estimated without smoothing or more training data.


# Part XIV – Connection to Modern Language Models

A modern autoregressive language model also targets:

`P(Xt | X1,...,Xt-1)`.

The key difference is how this conditional distribution is represented:

| Simple BN / n-gram model | Modern autoregressive LM |
|---|---|
| CPTs | Neural network |
| Fixed context window | Large learned context |
| Explicit probabilities | Learned weights |
| Counting | Gradient-based training |
| Sampling for generation | Sampling for generation |

The laboratory explicitly warns that first- and second-order n-gram models are not comparable in expressive power to modern LLMs. Their important commonality is the probabilistic objective: predicting the next token conditional on previous tokens.


# Question 13 – Reflection on LLM Use

**Approach B** (“Implement the following probabilistic model: `P(Xt|Xt-1)`, estimated from transition counts, with sampling-based generation”) is preferable because it specifies the intended behaviour before implementation.

1. **Specify intended behaviour:** the model's mathematical objective and generation procedure are explicit.
2. **Understand representation:** the programmer knows that the model is a transition-count/CPT model rather than an arbitrary neural language model.
3. **Validate implementation:** generated code can be checked against the specification.
4. **Test probabilistic invariants:** normalization tests verify that each conditional distribution sums to one.
5. **Distinguish implementation from model:** a Python program is only an implementation; correctness means that its behaviour matches the specified probability model.

This follows the laboratory's overall workflow: **Understand → Design → Ask the LLM → Implement → Test → Reflect**.


# Question 14 – What Did the Bayesian Network Add?

Thinking of the language model as a Bayesian network provides several useful perspectives:

1. **Dependency representation:** the graph explicitly shows which previous tokens a token depends on.
2. **Joint-distribution factorisation:** the chain rule becomes a product of local conditional probabilities.
3. **Interpretation of CPTs:** transition probabilities have a direct probabilistic meaning.
4. **Principled generation:** generation follows the same conditional distributions used by the factorisation.
5. **Independence assumptions:** the first-order graph makes the Markov assumption explicit.
6. **Context effects:** moving from one-token to two-token context changes the graph and the size/sparsity of the CPT.
7. **Implementation testing:** the probabilistic specification gives invariants such as conditional probabilities summing to one.

Thus the BN viewpoint connects the abstract probability model to the actual sequence-generation procedure. The laboratory explicitly asks students to discuss these connections.


# LLM-generated Code Inspection and Validation

One concrete implementation detail inspected independently was the transition-probability calculation:

`P(next | current) = transition_count / total_outgoing_count`.

The generated implementation was checked against the manually computed CPT. For example:

- `the → cat/dog`: `0.5 / 0.5`
- `cat → sat/ran`: `2/3 / 1/3`
- `dog → sat/ran`: `2/3 / 1/3`
- `sat → on`: `1.0`
- `ran → to`: `1.0`

The normalization test was also executed for every stored context. All recorded totals equal `1.0`.

This satisfies the laboratory's requirement to identify an LLM-assisted implementation component and independently validate it.


In [9]:
# Final validation summary
print("First-order normalization:", all(abs(v-1.0) < 1e-12 for v in first_norm.values()))
print("Second-order normalization:", all(abs(v-1.0) < 1e-12 for v in second_norm.values()))
print("First-order CPT entries:", first_params)
print("Second-order CPT entries:", second_params)
print("20-sentence first-order sample diversity:", div_first_sample, "/ 20")
print("20-sentence second-order sample diversity:", div_second_sample, "/ 20")
print("All required first-order contexts reported:", all(w in probabilities for w in ["the","cat","dog","sat","ran"]))


First-order normalization: True
Second-order normalization: True
First-order CPT entries: 17
Second-order CPT entries: 18
20-sentence first-order sample diversity: 13 / 20
20-sentence second-order sample diversity: 6 / 20
All required first-order contexts reported: True
